# 1_generate_data — regenerate the simulated data

This notebook regenerates the simulated behavioral and neural data used in the paper, starting from the trained stage-4 weights shipped in `saved_params/` (stage 4 is the final stage of the training curriculum; earlier stages are not needed for reproduction). It is the FIRST notebook to run: notebooks 2 and 3 read the files produced here.

## Data description

- **CUE** (`CUE_part{i}.pkl`): the two-stage random-dot-motion task. In stage 1 the agent decides between waiting and choosing; after a motion-coherence stimulus it commits to a direction in stage 2. The data frame contains per-trial behavior (coherence, correctness, choice, reaction times, aborts) and the full RNN activity (`fr`), states, actions, and choice probabilities for every time step.
- **CUEP** (`CUEP_part{i}.pkl`): CUE with a passive stage 1 — stage 1 is driven by the CUE-trained agent (loaded via `passive_agent_cfg`), while the CUEP-trained agent takes over for stage 2. The data frame additionally contains the passive agent's actions (`action_passive`) and choice probabilities (`chprob_passive`).
- **CUEP never-UR** (`CUEP_part{i}_never_ur.pkl`): control variant of CUEP where the passive agent never emits the uncertainty response (`never_ur=True`). This is the condition labeled "No Ext." in the paper figures.

Each part seeds the network-side randomness (`SEED_BASE + i`: RNN noise and action sampling). The task environment's randomness (trial conditions, period timing, stimulus noise) is not seeded, so reruns produce statistically equivalent but not bitwise-identical datasets (see the README's Reproducibility section). The paper generated 30 parts per condition and used the first 8 for the main analyses.

## Runtime

Measured on an Apple M3 Ultra (256 GB RAM, macOS 27, CPU-only): about **9 minutes** at the default scale (2 parts x 10000 trials per condition) and about **35 minutes** at the analysis scale (8 parts x 10000 trials). CUE runs at ~330 trials/s; the CUEP variants run at ~90-110 trials/s because of the additional passive-agent forward pass.

## Run order

Run this notebook **FIRST**, before `2_behavior_analysis.ipynb` and `3_neuron_ur_all_analysis.ipynb`.

In [ ]:
from pathlib import Path
import numpy as np
import torch
import pandas as pd
from data_load import DataCUE, DataCUEP
import config.hyperparams as hyperparams

**Analysis-scale reproduction (paper data):** set `N_PARTS=8` below (`N_TRIALS=10000` is already the default) — 8 parts x 10000 trials per condition, ~35 minutes on an Apple M3 Ultra. The default `N_PARTS=2` is a quick check that the pipeline runs end to end.

In [ ]:
N_PARTS = 2  # paper: 30 parts generated per condition, main analyses use the first 8
N_TRIALS = 10000  # trials per part, as in the paper
STAGE = 4  # final stage of the training curriculum
cohs = [2, 4, 6, 8, 13, 26, 52, 78]
SEED_BASE = 2024

save_dir = Path('data')
save_dir.mkdir(exist_ok=True)

In [ ]:
for i in range(N_PARTS):
    np.random.seed(SEED_BASE + i)
    torch.manual_seed(SEED_BASE + i)

    print(f"Generating CUE part {i + 1}/{N_PARTS} ...")
    df = DataCUE(
        hyperparams.hy_params,
        celltype='vanillazero',
        actype='wang',
        ppotype='classic',
    ).simulate(
        STAGE,
        N_TRIALS,
        mark='new',
        cohs=cohs,
    )
    df.to_pickle(save_dir / f"CUE_part{i}.pkl",
                 compression={"method": "gzip", "compresslevel": 4, "mtime": 1})
    print(f"Saved {save_dir / f'CUE_part{i}.pkl'} (shape: {df.shape})")

In [ ]:
# The passive agent driving stage 1 of CUEP is the CUE-trained network
passive_agent_cfg = {
    "hp": hyperparams.hy_params,
    "stage": 4,
    "mark": "new",
}

for i in range(N_PARTS):
    np.random.seed(SEED_BASE + i)
    torch.manual_seed(SEED_BASE + i)

    print(f"Generating CUEP part {i + 1}/{N_PARTS} ...")
    df = DataCUEP(
        hyperparams.hy_params_cuep,
        celltype='vanillazero',
        actype='wang',
        ppotype='classic',
    ).simulate(
        STAGE,
        N_TRIALS,
        mark='newcuep',
        cohs=cohs,
        passive_toldur=200,
        passive_agent_cfg=passive_agent_cfg,
    )
    df.to_pickle(save_dir / f"CUEP_part{i}.pkl",
                 compression={"method": "gzip", "compresslevel": 4, "mtime": 1})
    print(f"Saved {save_dir / f'CUEP_part{i}.pkl'} (shape: {df.shape})")

In [ ]:
for i in range(N_PARTS):
    np.random.seed(SEED_BASE + i)
    torch.manual_seed(SEED_BASE + i)

    print(f"Generating CUEP (never-UR) part {i + 1}/{N_PARTS} ...")
    df = DataCUEP(
        hyperparams.hy_params_cuep,
        celltype='vanillazero',
        actype='wang',
        ppotype='classic',
    ).simulate(
        STAGE,
        N_TRIALS,
        mark='newcuep',
        cohs=cohs,
        passive_toldur=200,
        passive_agent_cfg=passive_agent_cfg,
        never_ur=True,
    )
    df.to_pickle(save_dir / f"CUEP_part{i}_never_ur.pkl",
                 compression={"method": "gzip", "compresslevel": 4, "mtime": 1})
    print(f"Saved {save_dir / f'CUEP_part{i}_never_ur.pkl'} (shape: {df.shape})")

In [ ]:
print("Generated files:")
for path in sorted(save_dir.glob("*.pkl")):
    df = pd.read_pickle(path, compression="gzip")
    print(f"\n{path}")
    print(f"  shape: {df.shape}")
    print(f"  columns: {list(df.columns)}")